# Real-Time Fraud Prediction Publisher

This notebook consumes transactions from the Kafka topic:

fraud-transactions

predicts fraud using the trained Random Forest model

and publishes enriched prediction results to another Kafka topic:

fraud-predictions

This notebook acts as the real-time Machine Learning Prediction Engine.

## Imports

In [33]:
import json
import joblib
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

In [34]:
spark = (
    SparkSession.builder
    .appName("FraudPredictionPublisher")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

## Stop Existing Streams

In [35]:
for stream in spark.streams.active:
    stream.stop()

print("Existing streams stopped.")

ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/2s/z01cdgj13ydbbdpl84qrhzsr0000gn/T/ipykernel_50778/2077906673.py", line 12, in predict_batch
    pdf = batch_df.toPandas()
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fra

Existing streams stopped.


Exception in thread "stream execution thread for [id = 50b59753-5d67-4649-9db9-7d89433dc8a0, runId = 2eee4996-9e44-41f2-ad6d-3d7870931288]" java.lang.StackOverflowError
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:3954)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4761)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4816)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:4925)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:4847)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4725)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:3958)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4761)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4816)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:4925)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:4847)
	at java.base/java.util.regex.Pattern

## Load Best Model

In [36]:
model = joblib.load("../models/best_model.pkl")

print(model)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)


## Load Metadata

In [37]:
with open("../models/best_model_metadata.json","r") as f:
    metadata=json.load(f)

metadata

{'model_name': 'Random Forest (Class Weights)',
 'filename': 'best_model.pkl',
 'algorithm': 'RandomForestClassifier',
 'accuracy': 0.999489,
 'precision': 0.958333,
 'recall': 0.726316,
 'f1_score': 0.826347,
 'roc_auc': 0.943893,
 'random_state': 42,
 'scikit_learn_version': '1.9.0',
 'saved_on': '2026-07-11 20:13:28'}

## Read Kafka Stream

In [38]:
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","localhost:9092")
    .option("subscribe","fraud-transactions")
    .option("startingOffsets","latest")
    .load()
)

## Schema

In [39]:
schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("event_timestamp", StringType()),
    StructField("Time", DoubleType()),

    *[
        StructField(f"V{i}", DoubleType())
        for i in range(1,29)
    ],

    StructField("Amount", DoubleType()),
    StructField("Class", DoubleType())
])

## Parse JSON

In [40]:
parsed_df = (
    stream_df
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            schema
        ).alias("data")
    )
    .select("data.*")
)

## Verify Schema

In [41]:
parsed_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)


## Feature Engineering

In [42]:
feature_df = (
    parsed_df

    .withColumn(
        "Large_Transaction",
        when(col("Amount") > 200, 1).otherwise(0)
    )

    .withColumn(
        "Log_Amount",
        log1p(col("Amount"))
    )

    .withColumn(
        "Amount_Level",
        when(col("Amount") < 10, 0)
        .when(col("Amount") < 100, 1)
        .when(col("Amount") < 500, 2)
        .otherwise(3)
    )
)

## Prediction Function

In [43]:
from pyspark.sql import Row

def predict_batch(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return spark.createDataFrame([], schema="""
            transaction_id STRING,
            event_timestamp STRING,
            Amount DOUBLE,
            Prediction INT,
            Fraud_Probability DOUBLE,
            Status STRING,
            Risk_Level STRING
        """)

    X = pdf[model.feature_names_in_]

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:,1]

    pdf["Prediction"] = predictions
    pdf["Fraud_Probability"] = probabilities.round(6)

    pdf["Status"] = pdf["Prediction"].map({
        0:"Genuine",
        1:"Fraud"
    })

    def risk(prob):

        if prob>=0.90:
            return "Critical"

        elif prob>=0.70:
            return "High"

        elif prob>=0.40:
            return "Medium"

        else:
            return "Low"

    pdf["Risk_Level"]=pdf["Fraud_Probability"].apply(risk)

    rows=[]

    for _,row in pdf.iterrows():

        rows.append(Row(

            transaction_id=row["transaction_id"],

            event_timestamp=row["event_timestamp"],

            Amount=float(row["Amount"]),

            Prediction=int(row["Prediction"]),

            Fraud_Probability=float(row["Fraud_Probability"]),

            Status=row["Status"],

            Risk_Level=row["Risk_Level"]

        ))

    return spark.createDataFrame(rows)

## Output Schema

In [44]:
prediction_schema = StructType([

    StructField("transaction_id", StringType()),

    StructField("event_timestamp", StringType()),

    StructField("Amount", DoubleType()),

    StructField("Prediction", IntegerType()),

    StructField("Fraud_Probability", DoubleType()),

    StructField("Status", StringType()),

    StructField("Risk_Level", StringType())

])

## Apply Prediction

In [45]:
prediction_stream = (

    feature_df.writeStream

    .foreachBatch(
        lambda batch_df, batch_id:
        predict_batch(batch_df, batch_id)
    )

)

## Convert Predictions to JSON

In [46]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)


def predict_batch(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return

    # -------------------------
    # Arrange features exactly
    # -------------------------

    X = pdf[model.feature_names_in_]

    predictions = model.predict(X)

    probabilities = model.predict_proba(X)[:, 1]

    pdf["Prediction"] = predictions

    pdf["Fraud_Probability"] = probabilities.round(6)

    pdf["Status"] = pdf["Prediction"].map({
        0: "Genuine",
        1: "Fraud"
    })

    def risk(prob):

        if prob >= 0.90:
            return "Critical"

        elif prob >= 0.70:
            return "High"

        elif prob >= 0.40:
            return "Medium"

        else:
            return "Low"

    pdf["Risk_Level"] = pdf["Fraud_Probability"].apply(risk)

    print("\n")
    print("=" * 120)
    print(f"Publishing Batch {batch_id}")
    print("=" * 120)

    for _, row in pdf.iterrows():

        message = {

            "transaction_id": row["transaction_id"],

            "event_timestamp": row["event_timestamp"],

            "Amount": float(row["Amount"]),

            "Prediction": int(row["Prediction"]),

            "Fraud_Probability": float(row["Fraud_Probability"]),

            "Risk_Level": row["Risk_Level"],

            "Status": row["Status"]

        }

        producer.send(
            "fraud-predictions",
            value=message
        )

        print(message)

    producer.flush()

    print(f"\nPublished {len(pdf)} prediction(s) to Kafka.")

## Start Prediction Publisher

In [47]:
query = (

    feature_df

    .writeStream

    .foreachBatch(predict_batch)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../checkpoints/prediction_publisher"
    )

    .start()

)

## Keep Alive

In [48]:
query.awaitTermination() 



Publishing Batch 560
{'transaction_id': '92c52446-6681-40df-8d8e-256a91fe4928', 'event_timestamp': '2026-07-12T07:29:14.586+00:00', 'Amount': 40.8, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 561
{'transaction_id': 'd88e43da-a5f6-45b0-910b-be1903994367', 'event_timestamp': '2026-07-12T07:29:15.591+00:00', 'Amount': 93.2, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 562
{'transaction_id': '2b7519b3-bfbc-4d53-a9b9-7955e03195ac', 'event_timestamp': '2026-07-12T07:29:16.597+00:00', 'Amount': 3.68, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 563
{'transaction_id': '38142720-a53f-4365-8fa2-20afb487b1f9', 'event_timestamp': '2026-07-12T07:29:17.597+00:00', 'Amount': 7.8, 'Prediction': 0, 'Fraud_Probability



Publishing Batch 582
{'transaction_id': 'e8b28b07-13a3-40b1-92cb-729dc38c0f79', 'event_timestamp': '2026-07-12T07:29:36.693+00:00', 'Amount': 12.99, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 583
{'transaction_id': 'ebb9efef-bab1-4824-9feb-c738d57a06cf', 'event_timestamp': '2026-07-12T07:29:37.698+00:00', 'Amount': 17.28, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 584
{'transaction_id': '8832e47f-58f0-4c76-b2ae-eee1fce7a43c', 'event_timestamp': '2026-07-12T07:29:38.702+00:00', 'Amount': 4.45, 'Prediction': 0, 'Fraud_Probability': 0.01, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 585
{'transaction_id': '5086829d-2797-47a8-837f-f7a9fbcb88db', 'event_timestamp': '2026-07-12T07:29:39.707+00:00', 'Amount': 6.14, 'Prediction': 0, 'Fraud_Probabi



Publishing Batch 586
{'transaction_id': 'a25d3170-346e-49a6-a302-5bf0fc49f12c', 'event_timestamp': '2026-07-12T07:29:40.713+00:00', 'Amount': 6.14, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 587
{'transaction_id': 'a1aeb3c3-74e2-42e3-8a8a-460cf48a4607', 'event_timestamp': '2026-07-12T07:29:41.719+00:00', 'Amount': 1.77, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 588
{'transaction_id': '25b099aa-0f11-4b3d-90b7-8d65ed5b7681', 'event_timestamp': '2026-07-12T07:29:42.725+00:00', 'Amount': 1.77, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 589
{'transaction_id': '57645c08-1154-4fc7-aa51-1a41ad0d229c', 'event_timestamp': '2026-07-12T07:29:43.731+00:00', 'Amount': 30.49, 'Prediction': 0, 'Fraud_Probabili



Publishing Batch 629
{'transaction_id': '374ac2ff-dc17-4405-ac01-8f65b7aeda19', 'event_timestamp': '2026-07-12T07:30:23.922+00:00', 'Amount': 59.99, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.




Publishing Batch 630
{'transaction_id': '14ca835e-deea-482b-804f-6b2e8520276b', 'event_timestamp': '2026-07-12T07:30:24.924+00:00', 'Amount': 135.51, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.




Publishing Batch 631
{'transaction_id': '3a6673ac-46e5-483d-991e-41ac9a1f0356', 'event_timestamp': '2026-07-12T07:30:25.925+00:00', 'Amount': 9.79, 'Prediction': 0, 'Fraud_Probability': 0.01, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 632
{'transaction_id': 'f6f0d035-058e-4a1e-9506-edee08558c53', 'event_timestamp': '2026-07-12T07:30:26.931+00:00', 'Amount': 14.8, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.




Publishing Batch 633
{'transaction_id': 'a0fea82e-aeca-4c4a-88ae-a4e5ccc97888', 'event_timestamp': '2026-07-12T07:30:27.936+00:00', 'Amount': 1.98, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 634
{'transaction_id': 'c6afd494-461b-44c4-a395-5c3319b4188d', 'event_timestamp': '2026-07-12T07:30:28.940+00:00', 'Amount': 6.67, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 635
{'transaction_id': '239cb277-f54d-4f8b-97d0-ddbd10d7f872', 'event_timestamp': '2026-07-12T07:30:29.945+00:00', 'Amount': 1.46, 'Prediction': 0, 'Fraud_Probability': 0.0, 'Risk_Level': 'Low', 'Status': 'Genuine'}

Published 1 prediction(s) to Kafka.


Publishing Batch 636
{'transaction_id': 'adfea762-d680-4354-8fcb-e04e23d94e04', 'event_timestamp': '2026-07-12T07:30:30.950+00:00', 'Amount': 89.17, 'Prediction': 0, 'Fraud_Probabili

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 